# 03a — Preparación no supervisada (conjuntos A/B/C/D)

Ejecuta el §5 + primera comprobación §7A de planning/plan_unsupervised_implementation.md.
Nota de convención: las transformaciones de ≤5 líneas viven aquí por legibilidad; todo lo que puede romperse en silencio (listas, matemática CLC, guardianes, config) vive en src/wildfire/processing/unsupervised.py con tests en 	ests/test_unsupervised.py.
Entrada: data/processed/expanded/firms_spain_final_lags_index.csv (30.412×60). Nunca irms_spain_final_normalized.csv (scaler descriptivo, plan §4 regla 3).
Salida: data/processed/unsupervised/features_{a,b,c}.csv + perfilado.csv (índice común).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from wildfire.config import PROJECT_ROOT
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.pca_weather import WEATHER_PCA_BASE
from wildfire.processing.unsupervised import (
    ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED, ENTRADA_D, PERFILADO,
    add_wind_circular, apply_log_transforms, assert_feature_sets_clean,
    build_clc_environment, clc_environment_columns, resolve_set,
)

## 03a.1 — Cargar base y derivar viento circular

wd_sin/wd_cos se derivan al vuelo con AddWindCircular (igual que el PCA meteorológico); la columna circular cruda no entra en ningún conjunto.

In [ ]:
df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
print(f"{len(df):,} rows × {df.shape[1]} columns")
df = add_wind_circular(df)
assert {"wd_sin", "wd_cos"} <= set(df.columns)

## 03a.2 — Mostrar los conjuntos cerrados y vigilarlos

Las listas se importan (fuente única de verdad) y se imprimen aquí para leer el pipeline entero. El guardián falla en cerrado ante fuga de satélite (plan §4 regla 1).

In [ ]:
for _name, _cols in [("A", ENTRADA_A), ("B", ENTRADA_B),
                     ("C_fixed", ENTRADA_C_FIXED), ("D", ENTRADA_D)]:
    print(f"{_name} ({len(_cols)}): {_cols}")
print(f"PERFILADO ({len(PERFILADO)})")
assert_feature_sets_clean(ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED, ENTRADA_D)
print("Guard OK: no satellite columns in inputs.")

## 03a.3 — Transformaciones (≤5 líneas cada una, aquí por la nota de convención)

log1p en columnas asimétricas; BLH queda con mediana global por decisión (plan §3); entorno CLC colapsado con el constructor con tests.

In [ ]:
df = apply_log_transforms(df)
env = build_clc_environment(df)
df = pd.concat([df, env], axis=1)
env_cols = clc_environment_columns(env)
print(f"CLC environment: {len(env_cols)} columns ({len([c for c in env_cols if c.startswith('freq_vecina_')])} frequencies)")
display(env.describe().T[["mean", "std", "min", "max"]])

## 03a.4 — Resolver conjuntos y guardar (índice común)

Las columnas ausentes avisan y se descartan (fallo en cerrado); los cuatro ficheros comparten el índice para re-unir.

In [ ]:
sets = {
    "features_a": resolve_set(df, ENTRADA_A, "A"),
    "features_b": resolve_set(df, ENTRADA_B, "B"),
    "features_c": resolve_set(df, [*ENTRADA_C_FIXED, *env_cols], "C"),
    "perfilado": resolve_set(df, PERFILADO, "PERFILADO"),
}
for _name, _frame in sets.items():
    save_csv(_frame, f"data/processed/unsupervised/{_name}.csv")
    print(f"{_name}: {_frame.shape}")

## 03a.5 — Primer número de robustez: ARI Standard vs Robust (plan §7A)

Mismo KMeans (k=5, semilla 42) sobre las 12 continuas meteorológicas con ambos scalers. Es una sonda de robustez al preprocesado, no una elección de k (eso es 03b/03e).

In [ ]:
base12 = resolve_set(df, WEATHER_PCA_BASE, "BASE12")
std_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(StandardScaler().fit_transform(base12))
rob_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(RobustScaler().fit_transform(base12))
ari = adjusted_rand_score(std_labels, rob_labels)
print(f"ARI(Standard, Robust) @ k=5: {ari:.3f} (plan §7A bar: > 0.6 across variants)")